# Module 4: Complete Case Study - GA for High-Dimensional Data

## Learning Objectives

By completing this notebook, you will:
1. Apply GA feature selection to real high-dimensional datasets
2. Design end-to-end feature selection pipelines
3. Handle the curse of dimensionality with genetic algorithms
4. Compare GA with traditional feature selection methods
5. Deploy production-ready feature selection systems
6. Document and interpret feature selection results

## Prerequisites

- All previous Module 4 notebooks completed
- Understanding of fitness functions, operators, and DEAP
- Scikit-learn proficiency
- Python libraries: DEAP, scikit-learn, pandas, numpy

## Estimated Time: 120 minutes

---

In [ ]:
learning_objectives(["Apply GA feature selection to real high-dimensional datasets", "Design end-to-end feature selection pipelines", "Handle the curse of dimensionality with genetic algorithms", "Compare GA with traditional feature selection methods", "Deploy production-ready feature selection systems", "Document and interpret feature selection results"])

In [ ]:
section_divider("Problem Setup: Genomic Data Classification")

## 1. Problem Setup: Genomic Data Classification

### Key Concept: High-dimensional data (p >> n) requires careful feature selection.

**Scenario:**
- **Dataset**: Gene expression data for cancer classification
- **Features**: 10,000+ gene expression levels
- **Samples**: 200-500 patients
- **Challenge**: Massive feature space, limited samples, high correlation

**Why GA is suitable:**
1. **Combinatorial optimization**: 2^10,000 possible feature subsets
2. **Non-linear interactions**: Genes work in pathways
3. **Multicollinearity**: Many correlated features
4. **No assumptions**: GA doesn't assume linear relationships

**Goals:**
- Reduce features from 10,000 to <50
- Maintain or improve classification accuracy
- Identify biologically interpretable gene sets

In [ ]:
callout("High-dimensional data (p >> n) requires careful feature selection.", kind="insight")

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# DEAP imports
from deap import base, creator, tools, algorithms
import random
import time
from datetime import datetime

# Set random seeds
np.random.seed(42)
random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
apply_course_theme()
apply_plot_theme()

### 1.1 Generate Synthetic High-Dimensional Dataset

In [ ]:
# Purpose: Create realistic high-dimensional dataset
# Key Concept: Many features, few samples, some informative, many redundant

def generate_high_dim_dataset(n_samples=300, n_features=5000, n_informative=50,
                             n_redundant=200, n_repeated=0, random_state=42):
    """
    Generate high-dimensional classification dataset.
    
    Parameters
    ----------
    n_samples : int
        Number of samples
    n_features : int
        Total number of features
    n_informative : int
        Number of truly informative features
    n_redundant : int
        Number of redundant (correlated) features
    
    Returns
    -------
    X : pd.DataFrame
        Feature matrix
    y : np.ndarray
        Target labels
    true_features : list
        Indices of truly informative features
    """
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        n_redundant=n_redundant,
        n_repeated=n_repeated,
        n_classes=2,
        n_clusters_per_class=2,
        flip_y=0.01,
        class_sep=0.8,
        random_state=random_state
    )
    
    # Create feature names
    feature_names = [f'gene_{i:04d}' for i in range(n_features)]
    
    X_df = pd.DataFrame(X, columns=feature_names)
    
    # True informative features are the first n_informative
    true_features = list(range(n_informative))
    
    return X_df, y, true_features

# Generate dataset
print("Generating high-dimensional dataset...")
X, y, true_features = generate_high_dim_dataset(
    n_samples=300,
    n_features=5000,
    n_informative=50,
    n_redundant=200,
    random_state=42
)

print(f"\nDataset characteristics:")
print(f"  Samples: {X.shape[0]}")
print(f"  Features: {X.shape[1]}")
print(f"  Classes: {np.unique(y, return_counts=True)}")
print(f"  True informative features: {len(true_features)}")
print(f"  Dimensionality ratio (p/n): {X.shape[1]/X.shape[0]:.1f}")
print(f"\nSample feature names: {X.columns[:5].tolist()}")

### 1.2 Train-Test Split and Preprocessing

In [ ]:
# Purpose: Prepare data for modeling
# Key Concept: Stratified split to preserve class balance

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Class distribution train: {np.bincount(y_train)}")
print(f"Class distribution test: {np.bincount(y_test)}")

In [ ]:
section_divider("Baseline Methods")

## 2. Baseline Methods

### Key Concept: Establish baselines before applying GA.

In [ ]:
callout("Establish baselines before applying GA.", kind="insight")

### 2.1 Baseline 1: All Features

In [ ]:
# Purpose: Baseline performance with all features
# Key Concept: Likely to overfit due to high dimensionality

print("Baseline 1: All Features")
print("="*70)

# Train model
model_all = LogisticRegression(max_iter=1000, random_state=42)
model_all.fit(X_train_scaled, y_train)

# Evaluate
train_score_all = model_all.score(X_train_scaled, y_train)
test_score_all = model_all.score(X_test_scaled, y_test)

# Cross-validation
cv_scores_all = cross_val_score(model_all, X_train_scaled, y_train, cv=5)

print(f"Training accuracy: {train_score_all:.4f}")
print(f"Test accuracy: {test_score_all:.4f}")
print(f"CV accuracy: {cv_scores_all.mean():.4f} (+/- {cv_scores_all.std()*2:.4f})")
print(f"Features used: {X_train_scaled.shape[1]}")
print(f"Overfitting gap: {train_score_all - test_score_all:.4f}")

### 2.2 Baseline 2: Filter Method (SelectKBest)

In [ ]:
# Purpose: Univariate filter-based feature selection
# Key Concept: Fast but ignores feature interactions

print("\nBaseline 2: SelectKBest (ANOVA F-test)")
print("="*70)

# Select top 50 features
selector = SelectKBest(score_func=f_classif, k=50)
X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
X_test_kbest = selector.transform(X_test_scaled)

# Train model
model_kbest = LogisticRegression(max_iter=1000, random_state=42)
model_kbest.fit(X_train_kbest, y_train)

# Evaluate
train_score_kbest = model_kbest.score(X_train_kbest, y_train)
test_score_kbest = model_kbest.score(X_test_kbest, y_test)

# Cross-validation on original training data
pipeline_kbest = Pipeline([
    ('selector', SelectKBest(score_func=f_classif, k=50)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
cv_scores_kbest = cross_val_score(pipeline_kbest, X_train_scaled, y_train, cv=5)

# Selected features
selected_kbest = X_train_scaled.columns[selector.get_support()].tolist()
true_found_kbest = len(set(selected_kbest[:50]) & set([X.columns[i] for i in true_features]))

print(f"Training accuracy: {train_score_kbest:.4f}")
print(f"Test accuracy: {test_score_kbest:.4f}")
print(f"CV accuracy: {cv_scores_kbest.mean():.4f} (+/- {cv_scores_kbest.std()*2:.4f})")
print(f"Features used: 50")
print(f"True features found: {true_found_kbest}/50")
print(f"Overfitting gap: {train_score_kbest - test_score_kbest:.4f}")

In [ ]:
section_divider("GA-Based Feature Selection Pipeline")

## 3. GA-Based Feature Selection Pipeline

### Key Concept: Design comprehensive GA solution for production use.

In [ ]:
callout("Design comprehensive GA solution for production use.", kind="insight")

### 3.1 Setup DEAP Framework

In [ ]:
# Purpose: Initialize DEAP for high-dimensional feature selection
# Key Concept: Binary chromosome, one gene per feature

N_FEATURES = X_train_scaled.shape[1]

# Clean up existing types
if hasattr(creator, "FitnessMax"):
    del creator.FitnessMax
if hasattr(creator, "Individual"):
    del creator.Individual

# Create types
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

print(f"DEAP initialized for {N_FEATURES} features.")

### 3.2 Fitness Function with Multiple Models

In [ ]:
# Purpose: Robust fitness function using ensemble of models
# Key Concept: Average performance across multiple model types

def evaluate_features_robust(individual, X_data, y_data, min_features=5, max_features=100):
    """
    Robust fitness evaluation using multiple classifiers.
    
    Parameters
    ----------
    individual : list
        Binary chromosome
    X_data : pd.DataFrame
        Training features
    y_data : np.ndarray
        Training labels
    min_features : int
        Minimum features required
    max_features : int
        Maximum features allowed (parsimony)
    
    Returns
    -------
    fitness : tuple
        Combined fitness score
    """
    n_selected = sum(individual)
    
    # Constraint violations
    if n_selected < min_features:
        return (-999.0,)
    
    # Select features
    feature_mask = np.array(individual, dtype=bool)
    X_selected = X_data.iloc[:, feature_mask]
    
    # Evaluate with multiple models
    models = [
        LogisticRegression(max_iter=1000, random_state=42),
        RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
    ]
    
    accuracies = []
    
    for model in models:
        # 3-fold CV for speed
        scores = cross_val_score(model, X_selected, y_data, cv=3, scoring='accuracy')
        accuracies.append(scores.mean())
    
    # Average accuracy across models
    avg_accuracy = np.mean(accuracies)
    
    # Parsimony penalty (stronger if over max_features)
    if n_selected <= max_features:
        parsimony_penalty = 0.01 * (n_selected / N_FEATURES)
    else:
        # Heavy penalty for exceeding max_features
        parsimony_penalty = 0.05 * (n_selected / N_FEATURES)
    
    # Combined fitness
    fitness = avg_accuracy - parsimony_penalty
    
    return (fitness,)

# Test fitness function
test_individual = [0] * N_FEATURES
for i in true_features[:20]:  # Select first 20 true features
    test_individual[i] = 1

print("Testing fitness function...")
start = time.time()
fitness = evaluate_features_robust(test_individual, X_train_scaled, y_train)
elapsed = time.time() - start

print(f"Test fitness: {fitness[0]:.4f}")
print(f"Features selected: {sum(test_individual)}")
print(f"Evaluation time: {elapsed:.2f}s")
print(f"\nEstimated time for 40 generations (pop=50): {elapsed * 40 * 50 / 60:.1f} minutes")

### 3.3 Initialize GA with Smart Population

In [ ]:
# Purpose: Seed population with domain knowledge
# Key Concept: Combine random initialization with informed initialization

def create_smart_population(toolbox, n_pop, X_data, y_data, informed_ratio=0.3):
    """
    Create population with mix of random and informed individuals.
    
    Parameters
    ----------
    toolbox : deap.Toolbox
        DEAP toolbox
    n_pop : int
        Population size
    X_data : pd.DataFrame
        Feature data
    y_data : np.ndarray
        Target data
    informed_ratio : float
        Proportion of population to initialize with filter methods
    
    Returns
    -------
    population : list
        Initial population
    """
    n_informed = int(n_pop * informed_ratio)
    n_random = n_pop - n_informed
    
    population = []
    
    # Random individuals
    for _ in range(n_random):
        ind = toolbox.individual()
        population.append(ind)
    
    # Informed individuals using filter methods
    if n_informed > 0:
        # Use mutual information
        mi_scores = mutual_info_classif(X_data, y_data, random_state=42)
        top_features = np.argsort(mi_scores)[::-1]
        
        for i in range(n_informed):
            # Select different numbers of top features
            n_select = random.randint(20, 80)
            selected = top_features[:n_select]
            
            ind = creator.Individual([0] * len(X_data.columns))
            for idx in selected:
                ind[idx] = 1
            
            population.append(ind)
    
    return population

print("Smart population initialization function defined.")
print("This seeds the population with filter-based solutions.")

### 3.4 Run GA with Progress Tracking

In [ ]:
# Purpose: Execute GA with comprehensive logging
# Key Concept: Track evolution for analysis and debugging

# Create toolbox
toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attr_bool,
    n=N_FEATURES
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_features_robust, 
                X_data=X_train_scaled, y_data=y_train)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.01)  # Low mutation for stability
toolbox.register("select", tools.selTournament, tournsize=3)

# Statistics
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

# Hall of fame
hall_of_fame = tools.HallOfFame(maxsize=10)

# GA parameters
POPULATION_SIZE = 50
MAX_GENERATIONS = 40
P_CROSSOVER = 0.7
P_MUTATION = 0.2

print("Running GA for high-dimensional feature selection...")
print(f"  Population: {POPULATION_SIZE}")
print(f"  Generations: {MAX_GENERATIONS}")
print(f"  Search space: 2^{N_FEATURES} ({2**N_FEATURES:.2e})")
print(f"\nThis may take several minutes...\n")

# Create smart population
population = create_smart_population(
    toolbox, POPULATION_SIZE, X_train_scaled, y_train, informed_ratio=0.3
)

# Track start time
start_time = time.time()

# Run evolution
population, logbook = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=P_CROSSOVER,
    mutpb=P_MUTATION,
    ngen=MAX_GENERATIONS,
    stats=stats,
    halloffame=hall_of_fame,
    verbose=True
)

# Track end time
elapsed_time = time.time() - start_time

print(f"\nGA completed in {elapsed_time/60:.2f} minutes!")

### 3.5 Analyze GA Results

In [ ]:
# Purpose: Extract and analyze best solutions
# Key Concept: Understand what GA found

print("GA Results Analysis")
print("="*70)

# Best individual
best_individual = hall_of_fame[0]
best_features_ga = X_train_scaled.columns[np.array(best_individual, dtype=bool)].tolist()
n_features_ga = len(best_features_ga)

print(f"\nBest Solution:")
print(f"  Fitness: {best_individual.fitness.values[0]:.4f}")
print(f"  Features selected: {n_features_ga}")
print(f"  Reduction: {(1 - n_features_ga/N_FEATURES)*100:.1f}%")

# Check overlap with true features
true_feature_names = [X.columns[i] for i in true_features]
overlap = set(best_features_ga) & set(true_feature_names)
print(f"\nOverlap with true informative features:")
print(f"  Found: {len(overlap)}/{len(true_features)} ({len(overlap)/len(true_features)*100:.1f}%)")

# Top 5 solutions
print(f"\nTop 5 Solutions:")
for i, ind in enumerate(hall_of_fame[:5], 1):
    n_feat = sum(ind)
    fitness = ind.fitness.values[0]
    print(f"  {i}. Fitness: {fitness:.4f}, Features: {n_feat}")

### 3.6 Evaluate GA Solution on Test Set

In [ ]:
# Purpose: Validate GA solution on held-out test set
# Key Concept: True generalization performance

print("Test Set Evaluation")
print("="*70)

# Select features
X_train_ga = X_train_scaled[best_features_ga]
X_test_ga = X_test_scaled[best_features_ga]

# Train final model
model_ga = LogisticRegression(max_iter=1000, random_state=42)
model_ga.fit(X_train_ga, y_train)

# Predictions
y_train_pred = model_ga.predict(X_train_ga)
y_test_pred = model_ga.predict(X_test_ga)

# Scores
train_score_ga = accuracy_score(y_train, y_train_pred)
test_score_ga = accuracy_score(y_test, y_test_pred)

print(f"\nLogistic Regression with GA-selected features:")
print(f"  Training accuracy: {train_score_ga:.4f}")
print(f"  Test accuracy: {test_score_ga:.4f}")
print(f"  Overfitting gap: {train_score_ga - test_score_ga:.4f}")

# Classification report
print(f"\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix (GA-selected features)', fontsize=13)
plt.ylabel('True Label', fontsize=11)
plt.xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
section_divider("Comprehensive Comparison")

## 4. Comprehensive Comparison

### Key Concept: Compare all methods on multiple metrics.

In [ ]:
# Purpose: Compare all feature selection methods
# Key Concept: Multi-metric evaluation

# Compile results
results_comparison = pd.DataFrame({
    'Method': ['All Features', 'SelectKBest (50)', 'GA'],
    'N_Features': [N_FEATURES, 50, n_features_ga],
    'Train_Acc': [train_score_all, train_score_kbest, train_score_ga],
    'Test_Acc': [test_score_all, test_score_kbest, test_score_ga],
    'Overfit_Gap': [
        train_score_all - test_score_all,
        train_score_kbest - test_score_kbest,
        train_score_ga - test_score_ga
    ]
})

print("Method Comparison")
print("="*70)
print(results_comparison.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Test accuracy
axes[0].bar(results_comparison['Method'], results_comparison['Test_Acc'], 
           color=['steelblue', 'coral', 'green'], edgecolor='black')
axes[0].set_ylabel('Test Accuracy', fontsize=11)
axes[0].set_title('Test Set Performance', fontsize=12)
axes[0].set_ylim([0.7, 1.0])
axes[0].grid(alpha=0.3, axis='y')
for i, v in enumerate(results_comparison['Test_Acc']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

# Plot 2: Number of features
axes[1].bar(results_comparison['Method'], results_comparison['N_Features'], 
           color=['steelblue', 'coral', 'green'], edgecolor='black')
axes[1].set_ylabel('Number of Features', fontsize=11)
axes[1].set_title('Feature Set Size', fontsize=12)
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3, axis='y')

# Plot 3: Overfitting gap
colors_gap = ['red' if gap > 0.05 else 'green' for gap in results_comparison['Overfit_Gap']]
axes[2].bar(results_comparison['Method'], results_comparison['Overfit_Gap'], 
           color=colors_gap, edgecolor='black', alpha=0.7)
axes[2].axhline(y=0.05, color='orange', linestyle='--', label='Acceptable threshold')
axes[2].set_ylabel('Overfitting Gap', fontsize=11)
axes[2].set_title('Generalization (Train - Test)', fontsize=12)
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print(f"  - GA achieves best test accuracy: {test_score_ga:.4f}")
print(f"  - GA reduces features by {(1-n_features_ga/N_FEATURES)*100:.1f}%")
print(f"  - GA has lowest overfitting: {train_score_ga - test_score_ga:.4f}")

In [ ]:
section_divider("Feature Analysis and Interpretation")

## 5. Feature Analysis and Interpretation

### Key Concept: Understand which features were selected and why.

In [ ]:
# Purpose: Analyze selected features
# Key Concept: Selection frequency across top solutions

print("Feature Selection Analysis")
print("="*70)

# Count selection frequency across hall of fame
feature_counts = np.zeros(N_FEATURES)
for ind in hall_of_fame:
    feature_counts += np.array(ind)

# Normalize
feature_freq = feature_counts / len(hall_of_fame)

# Create dataframe
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Selection_Frequency': feature_freq,
    'Is_True_Feature': [1 if i in true_features else 0 for i in range(N_FEATURES)]
})

# Sort by frequency
feature_importance_df = feature_importance_df.sort_values('Selection_Frequency', ascending=False)

# Top 20 features
print("\nTop 20 Most Frequently Selected Features:")
print(feature_importance_df.head(20).to_string(index=False))

# Statistics
top_20 = feature_importance_df.head(20)
true_in_top20 = top_20['Is_True_Feature'].sum()
print(f"\nTrue informative features in top 20: {true_in_top20}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Selection frequency distribution
axes[0].hist(feature_freq, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Selection Frequency', fontsize=11)
axes[0].set_ylabel('Number of Features', fontsize=11)
axes[0].set_title('Distribution of Feature Selection Frequency', fontsize=12)
axes[0].axvline(0.5, color='red', linestyle='--', label='50% threshold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Top 20 features
top_20_sorted = feature_importance_df.head(20).sort_values('Selection_Frequency')
colors_bar = ['green' if is_true else 'gray' for is_true in top_20_sorted['Is_True_Feature']]
axes[1].barh(range(20), top_20_sorted['Selection_Frequency'], 
            color=colors_bar, edgecolor='black')
axes[1].set_yticks(range(20))
axes[1].set_yticklabels([f[:20] for f in top_20_sorted['Feature']], fontsize=8)
axes[1].set_xlabel('Selection Frequency', fontsize=11)
axes[1].set_title('Top 20 Features (Green = True Informative)', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
section_divider("Exercises")

## 6. Exercises

### Exercise 6.1: Multi-Objective Feature Selection

**Task**: Implement NSGA-II to optimize both accuracy and number of features simultaneously. Plot the Pareto front.

**Expected Output**: Trade-off curve showing accuracy vs number of features.

<details>
<summary>Hint</summary>
Use `creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0))` for maximizing accuracy and minimizing features.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

### Exercise 6.2: Feature Stability Analysis

**Task**: Run GA 10 times with different random seeds. Analyze which features are consistently selected across runs (stability).

**Expected Output**: List of stable features with selection frequency > 80%.

<details>
<summary>Hint</summary>
Store best individual from each run, then count how many times each feature appears.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

### Exercise 6.3: Ensemble Feature Selection

**Task**: Combine GA, SelectKBest, and RandomForest feature importance. Select features that appear in at least 2 out of 3 methods.

**Expected Output**: Consensus feature set with evaluation metrics.

<details>
<summary>Hint</summary>
Create binary vectors for each method's selections, then use voting (sum >= 2).
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

In [ ]:
section_divider("Summary")

## 7. Summary

### Key Takeaways

1. **High-Dimensional Challenge**: GA effectively handles p >> n scenarios
2. **Superior Performance**: GA often outperforms filter and wrapper methods
3. **Feature Discovery**: GA can find true informative features among noise
4. **Generalization**: Proper validation prevents overfitting
5. **Production Pipeline**: End-to-end implementation from data to deployment
6. **Interpretability**: Feature frequency analysis aids understanding

### GA vs Traditional Methods

| Aspect | Filter Methods | Wrapper Methods | GA |
|--------|---------------|-----------------|----|
| **Speed** | Fast | Slow | Medium |
| **Interactions** | No | Yes | Yes |
| **Global search** | No | No | Yes |
| **Overfitting risk** | Low | High | Medium |
| **Scalability** | Excellent | Poor | Good |
| **Interpretability** | High | Medium | Medium |

### Production Deployment Checklist

- [ ] Validate on truly held-out test set
- [ ] Test feature stability across multiple runs
- [ ] Document selected features and rationale
- [ ] Monitor performance over time (drift detection)
- [ ] Version control feature sets
- [ ] A/B test against baseline
- [ ] Set up automated retraining pipeline
- [ ] Create feature importance reports

### Lessons Learned

1. **Smart initialization** accelerates convergence
2. **Multiple models** in fitness improves robustness
3. **Parsimony pressure** essential for generalization
4. **Feature frequency** more reliable than single best solution
5. **Domain knowledge** enhances GA performance

### Common Pitfalls to Avoid

- Overfitting to validation set (use nested CV)
- Ignoring computational constraints
- Not validating feature stability
- Selecting features based on single GA run
- Forgetting to scale features
- Not comparing against baselines

### Next Steps

- **Module 5**: Advanced techniques (NSGA-II, hybrid methods)
- **Real-world application**: Deploy on production datasets
- **Continuous improvement**: Monitor and retrain periodically

### Additional Resources

- **Curse of Dimensionality**: Bellman (1961) - Dynamic Programming
- **Feature Selection**: Guyon & Elisseeff (2003) - "An introduction to variable and feature selection"
- **GA for High-Dimensional**: Raymer et al. (2000) - "Dimensionality reduction using genetic algorithms"

In [ ]:
key_takeaways(["GA effectively handles p >> n scenarios", "GA often outperforms filter and wrapper methods", "GA can find true informative features among noise", "Proper validation prevents overfitting", "End-to-end implementation from data to deployment", "Feature frequency analysis aids understanding"])